# Produce data

- Mô phỏng quá trình generate và đẩy dữ liệu vào db của một cty
- Mỗi giây sẽ chèn tầm 17_000 record vào db

In [1]:
import sys
sys.path.append('/home/nblong/Dropbox/Project/ops/ml-ops/src')

import os
import time

import numpy as np
import pandas as pd
import s3fs
from dotenv import load_dotenv
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, TimestampType

from utils.db_connection import PostgresConnection

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [5]:
packages = [
    "org.apache.hadoop:hadoop-aws:3.4.2",
    "software.amazon.awssdk:bundle:2.29.52",
    "org.postgresql:postgresql:42.7.11",
]

spark = (
    SparkSession
    .builder
    .appName('produce data')
    .config("spark.sql.legacy.parquet.nanosAsLong", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config('spark.jars.packages', ','.join(packages))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    .getOrCreate()
)

spark

- Thông tin cột, kiểu dữ liệu của cột, mapping để đổi tên cột

In [6]:
LIST_COLS = [
    "vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count",
    "trip_distance", "rate_code_id", "store_and_fwd_flag", "pickup_longitude",
    "pickup_latitude", "dropoff_longitude", "dropoff_latitude", "pu_location_id",
    "do_location_id", "payment_type", "fare_amount", "extra", "mta_tax",
    "surcharge", "improvement_surcharge", "tip_amount", "tolls_amount",
    "total_amount", "congestion_surcharge", "airport_fee", "cbd_congestion_fee",
]

LIST_DTYPES = {
    "vendor_id":             "object",
    "pickup_datetime":       "datetime64[ns]",
    "dropoff_datetime":      "datetime64[ns]",
    "passenger_count":       "float64",
    "trip_distance":         "float64",
    "rate_code_id":          "float64",
    "store_and_fwd_flag":    "object",
    "pickup_longitude":      "float64",
    "pickup_latitude":       "float64",
    "dropoff_longitude":     "float64",
    "dropoff_latitude":      "float64",
    "pu_location_id":        "object",
    "do_location_id":        "object",
    "payment_type":          "object",
    "fare_amount":           "float64",
    "extra":                 "float64",
    "mta_tax":               "float64",
    "surcharge":             "float64",
    "improvement_surcharge": "float64",
    "tip_amount":            "float64",
    "tolls_amount":          "float64",
    "total_amount":          "float64",
    "congestion_surcharge":  "float64",
    "airport_fee":           "float64",
    "cbd_congestion_fee":    "float64",
}

map1 = {
    "columns": 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "payment_type": "payment_type",
        "fare_amount": "fare_amount",
    }
}

map2 = {
    'columns':  'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "Airport_fee": "airport_fee",
    }
}

map3 = {
    'columns': 'vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_DateTime|Passenger_Count|Trip_Distance|Start_Lon|Start_Lat|Rate_Code|store_and_forward|End_Lon|End_Lat|Payment_Type|Fare_Amt|surcharge|mta_tax|Tip_Amt|Tolls_Amt|Total_Amt',
    'mapping': {
        "vendor_name": "vendor_id",
        "Trip_Pickup_DateTime": "pickup_datetime",
        "Trip_Dropoff_DateTime": "dropoff_datetime",
        "Passenger_Count": "passenger_count",
        "Trip_Distance": "trip_distance",
        "Start_Lon": "pickup_longitude",
        "Start_Lat": "pickup_latitude",
        "Rate_Code": "rate_code_id",
        "store_and_forward": "store_and_fwd_flag",
        "End_Lon": "dropoff_longitude",
        "End_Lat": "dropoff_latitude",
        "Payment_Type": "payment_type",
        "Fare_Amt": "fare_amount",
        "Tip_Amt": "tip_amount",
        "Tolls_Amt": "tolls_amount",
        "Total_Amt": "total_amount",
    }
}

map4 = {
    'columns': 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "Airport_fee": "airport_fee",
    }
}

map5 = {
    'columns':  'vendor_id|pickup_datetime|dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|rate_code|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount',
    'mapping': {
        "rate_code": "rate_code_id",
    }
}

map6 = {
    'columns':  'vendorID|tpepPickupDateTime|tpepDropoffDateTime|passengerCount|tripDistance|puLocationId|doLocationId|startLon|startLat|endLon|endLat|rateCodeId|storeAndFwdFlag|paymentType|fareAmount|extra|mtaTax|improvementSurcharge|tipAmount|tollsAmount|totalAmount',
    'mapping': {
        "vendorID": "vendor_id",
        "tpepPickupDateTime": "pickup_datetime",
        "tpepDropoffDateTime": "dropoff_datetime",
        "passengerCount": "passenger_count",
        "tripDistance": "trip_distance",
        "puLocationId": "pu_location_id",
        "doLocationId": "do_location_id",
        "startLon": "pickup_longitude",
        "startLat": "pickup_latitude",
        "endLon": "dropoff_longitude",
        "endLat": "dropoff_latitude",
        "rateCodeId": "rate_code_id",
        "storeAndFwdFlag": "store_and_fwd_flag",
        "paymentType": "payment_type",
        "fareAmount": "fare_amount",
        "mtaTax": "mta_tax",
        "improvementSurcharge": "improvement_surcharge",
        "tipAmount": "tip_amount",
        "tollsAmount": "tolls_amount",
        "totalAmount": "total_amount",
    }
}

map7 = {
    'columns': 'vendor_id|pickup_datetime|dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|rate_code|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount|__index_level_0__',
    'mapping': {
        "rate_code": "rate_code_id",
    }
}

SCHEMA_MAPPINGS = [map1, map2, map3, map4, map5, map6, map7]

- Tạo bảng

In [5]:
def create_table(conn):
    """Tạo bảng nyc_taxi nếu chưa tồn tại."""
    query = """
    CREATE TABLE IF NOT EXISTS public.nyc_taxi (
        vendor_id               TEXT,
        pickup_datetime         TIMESTAMP,
        dropoff_datetime        TIMESTAMP,
        passenger_count         FLOAT,
        trip_distance           FLOAT,
        rate_code_id            FLOAT,
        store_and_fwd_flag      TEXT,
        pickup_longitude        FLOAT,
        pickup_latitude         FLOAT,
        dropoff_longitude       FLOAT,
        dropoff_latitude        FLOAT,
        pu_location_id          TEXT,
        do_location_id          TEXT,
        payment_type            TEXT,
        fare_amount             FLOAT,
        extra                   FLOAT,
        mta_tax                 FLOAT,
        surcharge               FLOAT,
        improvement_surcharge   FLOAT,
        tip_amount              FLOAT,
        tolls_amount            FLOAT,
        total_amount            FLOAT,
        congestion_surcharge    FLOAT,
        airport_fee             FLOAT,
        cbd_congestion_fee      FLOAT
    );
    """
    with conn.cursor() as cur:
        cur.execute(query)
    conn.commit()
    print("Table 'nyc_taxi' created successfully.")

- Transform

In [15]:
SPARK_DTYPES = {
    "object": StringType(),
    "float64": DoubleType(),
    "datetime64[ns]": TimestampType(),
}

def _resolve_mapping(df_columns: str) -> dict:
    """Tìm schema mapping khớp với columns của file."""
    for schema in SCHEMA_MAPPINGS:
        if schema["columns"] == df_columns:
            return schema["mapping"]
    raise ValueError(f"No mapping found for {df_columns}")

def transform_data(df: DataFrame) -> DataFrame:
    """
    1. Đổi tên cột theo mapping tương ứng.
    2. Thêm cột canonical còn thiếu (fill null).
    3. Đổi kiểu dữ liệu về canonical dtype.
    4. Chỉ giữ lại các cột canonical.
    """
    # 1. Resolve & rename
    mapping = _resolve_mapping('|'.join(df.columns))
    for old_name, new_name in mapping.items():
        df = df.withColumnRenamed(old_name, new_name)

    # 2. Thêm cột còn thiếu
    for col in LIST_COLS:
        if col not in df.columns:
            df = df.withColumn(col, F.lit(None))

    # 3. Cast dtype
    for col, dtype in LIST_DTYPES.items():
        if 'datetime' in col and df.select(col).dtypes[0][1]=='bigint':
            # convert from nano-second to datetime
            # ref: https://stackoverflow.com/questions/72871432/convert-nanosecond-value-into-datetime-using-pyspark-in-databricks
            df = df.withColumn(col, F.to_timestamp(F.from_unixtime(F.col(col)/1000000000), 'yyyy-MM-dd HH:mm:ss'))
        else:
            spark_type = SPARK_DTYPES[dtype]
            df = df.withColumn(col, F.col(col).cast(spark_type))

    # 4. Chỉ giữ các cột canonical
    return df.select(LIST_COLS)

---
- Read data and transform

In [ ]:
# Source - https://stackoverflow.com/a/79434656
# Posted by Bas Kuis
# Retrieved 2026-07-21, License - CC BY-SA 4.0

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")


In [19]:
df = spark.read.parquet('../../../../data/yellow_tripdata_2020-02.parquet')
df.count()

6299367

In [17]:
df.select('tpepPickupDateTime').dtypes[0][1]

'bigint'

In [28]:
transform_data(df).show()

+---------+-------------------+-------------------+---------------+-------------+------------+------------------+----------------+---------------+-----------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|rate_code_id|store_and_fwd_flag|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+---------+-------------------+-------------------+---------------+-------------+------------+------------------+----------------+---------------+-----------------+----------------+--------------+--------------+------------+-----------+-----+-------+------

In [11]:
df = df.withColumn("tpepPickupDateTime", F.from_unixtime(F.col('tpepPickupDateTime')/1000000000))
# df = df.withColumn('pickup_datetime', F.from_unixtime(F.col('pickup_datetime_second')))
df.show()

+--------+-------------------+-------------------+--------------+------------+------------+------------+--------+--------+------+------+----------+---------------+-----------+----------+-----+------+--------------------+---------+-----------+-----------+----------------------+-------------------+
|vendorID| tpepPickupDateTime|tpepDropoffDateTime|passengerCount|tripDistance|puLocationId|doLocationId|startLon|startLat|endLon|endLat|rateCodeId|storeAndFwdFlag|paymentType|fareAmount|extra|mtaTax|improvementSurcharge|tipAmount|tollsAmount|totalAmount|pickup_datetime_second|    pickup_datetime|
+--------+-------------------+-------------------+--------------+------------+------------+------------+--------+--------+------+------+----------+---------------+-----------+----------+-----+------+--------------------+---------+-----------+-----------+----------------------+-------------------+
|       2|2020-01-03 03:57:12|1578024157000000000|             1|        1.03|          97|          97|  

In [8]:
df_ = spark.read.parquet('../../../../data/test/parquet/yellow_tripdata_2019-03.parquet')
df_.show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-03-01 00:24:41|  2019-03-01 00:25:31|            1.0|          0.0|       1.0|                 N|         145|         145|           2|        2.5|  0.5|    0.5|       0.

In [12]:
# read parquet
filepath = "s3://nblong-data-project/nyc_taxi/nyc-taxi/yellow_tripdata_2020-02.parquet"
df:DataFrame = spark.read.parquet(filepath)

# get 140_000 samples (asc pickup-date)
df = transform_data(df.sample(min(140_000/df.count(), 1.0)))
df.count()


139852

- Để có thể sort thì cần phải đánh index cho data
- Để có thể đánh index thì phải gom hết data vào 1 partition. Việc này nguy hiểm nếu dữ liệu quá lớn
- Tuy nhiên, đây chỉ là mô phỏng quá trình đẩy dữ liệu từ từ vào db, và dữ liệu cũng không quá lớn nên ok

In [7]:
batch_size = 17_000

# add an index column
window_spec = Window.orderBy('pickup_datetime')
df = df.withColumn('index', F.row_number().over(window_spec) - 1)
df.createOrReplaceTempView('nyc_taxi')

total_rows = df.count()
total_rows

26/05/20 09:29:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


140018

In [8]:
df.show(10)

26/05/20 09:29:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:29:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:29:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+-----------------+--------------------+-----------+------------------+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|     total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|index|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------

In [ ]:
for i in range(17000, total_rows, batch_size):
    start_index = i
    end_index = min(i+batch_size, total_rows-1)
    print(f'Process from {start_index} to {end_index}')

    tmp_df = spark.sql(f'''
        select *
        from nyc_taxi
        where (nyc_taxi.index >= {start_index}) and (nyc_taxi.index < {end_index})
    ''')
    tmp_df.drop('index').write \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://localhost:5432/company") \
        .option("dbtable", "public.nyc_taxi") \
        .option("user", "company") \
        .option("password", "company") \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()
    break

Process from 17000 to 34000


26/05/20 09:31:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:31:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:31:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [13]:
pd_df = tmp_df.toPandas()
pd_df.shape

26/05/20 09:32:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


(17000, 26)

In [14]:
pd_df.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_longitude,pickup_latitude,dropoff_longitude,...,mta_tax,surcharge,improvement_surcharge,tip_amount,tolls_amount,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,index
0,VTS,2010-01-05 13:43:00,2010-01-05 13:55:00,1.0,0.75,1.0,None,-73.968263,40.762598,-73.977300,...,0.5,0.0,NaN,0.00,0.0,7.80,NaN,NaN,NaN,17000
1,CMT,2010-01-05 13:43:17,2010-01-05 13:47:23,1.0,0.80,1.0,0,-73.993428,40.727361,-73.987621,...,0.5,0.0,NaN,0.75,0.0,5.75,NaN,NaN,NaN,17001
2,DDS,2010-01-05 13:43:30,2010-01-05 13:49:04,2.0,0.80,1.0,None,-73.971685,40.760172,-73.992723,...,-1.0,0.0,NaN,0.00,0.0,5.40,NaN,NaN,NaN,17002
3,CMT,2010-01-05 13:43:53,2010-01-05 13:50:04,3.0,1.00,1.0,0,-73.994666,40.741344,-73.988374,...,0.5,0.0,NaN,0.00,0.0,5.80,NaN,NaN,NaN,17003
4,VTS,2010-01-05 13:44:00,2010-01-05 14:23:00,1.0,17.58,1.0,None,-73.987490,40.737768,-73.782715,...,0.5,0.0,NaN,0.00,5.5,51.00,NaN,NaN,NaN,17004


In [16]:
tmp_df.show()

+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------------+-----+-------+---------+---------------------+----------+------------+-----------------+--------------------+-----------+------------------+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|      fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|     total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|index|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--

In [20]:
df = transform_data(df.sample(0.1, seed=42))
print(df.count())
df_ = (
    df
    .withColumn('source_file', F.lit('yellow_tripdata_2020-02.parquet'))
    .withColumn('year', F.substring('source_file', 17, 4).cast('int'))
    .withColumn('month', F.substring('source_file', 22, 2).cast('int'))
    .select('*')
    .where((F.col('year')==F.year(F.col('pickup_datetime'))) & (F.col('month')==F.month(F.col('pickup_datetime'))))
)
print(df_.count())

630237
630204


In [25]:
df__ = (
    df
    .withColumn('source_file', F.lit('yellow_tripdata_2020-02.parquet'))
    .withColumn('year', F.substring('source_file', 17, 4).cast('int'))
    .withColumn('month', F.substring('source_file', 22, 2).cast('int'))
    .select('*')
    .where((F.col('year')!=F.year(F.col('pickup_datetime'))) | (F.col('month')!=F.month(F.col('pickup_datetime'))))
)
df__.show()

+---------+-------------------+-------------------+---------------+-------------+------------+------------------+----------------+---------------+-----------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+--------------------+----+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|rate_code_id|store_and_fwd_flag|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|         source_file|year|month|
+---------+-------------------+-------------------+---------------+-------------+------------+------------------+----------------+---------------+-----------------+----------------+-----------

In [17]:
df.show()

26/09/22 20:14:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------------+-------------------+---------------+-------------+------------+------------------+----------------+---------------+-----------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+--------------------+----+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|rate_code_id|store_and_fwd_flag|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|         source_file|year|month|
+---------+-------------------+-------------------+---------------+-------------+------------+------------------+----------------+---------------+-----------------+----------------+-----------